# Merchant Fraud Risk Profile

This notebook covers **Member 4 fraud risk only**. It does not calculate a business score, tune ranking weights, or produce a Top 100. The primary output combines consumer exposure with a cross-validated KNN merchant-risk score.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

def find_member4_dir(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        direct = candidate if candidate.name == 'member4_fraud' else candidate / 'member4_fraud'
        if (direct / 'code').is_dir():
            return direct
    raise FileNotFoundError('Run this notebook from the repository or member4_fraud/code.')

MEMBER4_DIR = find_member4_dir()
CODE_DIR = MEMBER4_DIR / 'code'
sys.path.insert(0, str(CODE_DIR))

from build_fraud_risk_profile import build_fraud_risk_profile

REPO_ROOT = MEMBER4_DIR.parent
OUTPUT_DIR = MEMBER4_DIR / 'result'
RAW_TABLES = Path(os.environ.get('MEMBER4_RAW_TABLES', REPO_ROOT.parent / 'data' / 'part1' / 'tables'))
CURATED_TRANSACTIONS = Path(os.environ.get('MEMBER4_CURATED_TRANSACTIONS', REPO_ROOT / 'member2_curation' / 'data' / 'curated' / 'curated_transactions'))
summary = build_fraud_risk_profile(
    repo_root=REPO_ROOT, raw_tables_root=RAW_TABLES, output_dir=OUTPUT_DIR,
    curated_transactions_root=CURATED_TRANSACTIONS
)
display(summary)

,metric,value
0,total_merchants,4422.000000
1,merchants_with_consumer_risk,4414.000000
2,merchants_with_direct_fraud,61.000000
3,merchants_with_knn_risk,4422.000000
4,consumer_and_knn,4414.000000
5,consumer_only,0.000000
6,knn_only,8.000000
7,no_information,0.000000
8,knn_out_of_distribution,1763.000000
9,median_knn_confidence,0.098361


## 1. Consumer risk exposure

For merchant $m$, consumer exposure is the amount-weighted average consumer risk:

$$C_m = \frac{\sum_{t \in T_m^*} Amount_t \cdot ConsumerRisk_{u(t)}}{\sum_{t \in T_m^*} Amount_t}$$

It is converted to a merchant percentile $C_m^*$. Transaction and amount coverage are retained as reliability indicators but are not included in the score formula.

In [2]:
profile = pd.read_csv(OUTPUT_DIR / 'merchant_fraud_risk_profile.csv')
status_counts = profile['fraud_evidence_status'].value_counts(dropna=False).rename_axis('status').reset_index(name='merchants')
coverage_columns = [
    'consumer_risk_transaction_coverage', 'consumer_risk_amount_coverage',
    'observed_consumer_label_transaction_coverage',
    'observed_consumer_label_amount_coverage'
]
coverage_summary = profile[coverage_columns].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).T
display(status_counts, coverage_summary)

,status,merchants
0,consumer_and_knn,4414
1,knn_only,8


,count,mean,std,min,10%,25%,50%,75%,90%,max
consumer_risk_transaction_coverage,4414.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
consumer_risk_amount_coverage,4414.0,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.0
observed_consumer_label_transaction_coverage,4414.0,0.848155,0.065772,0.000000,0.800000,0.825204,0.837753,0.863397,0.928571,1.0
observed_consumer_label_amount_coverage,4411.0,0.853742,0.067893,0.297025,0.797702,0.824918,0.840039,0.878306,0.954145,1.0


## 2. KNN merchant-risk score

The 61 directly observed merchants provide the KNN training targets. After median imputation and StandardScaler, repeated cross-validation chooses K and neighbour weighting. The fitted KNN scores all eligible merchants, and its relative signal is:

$$K_m = Percentile(KNNPredictedMerchantFraud_m)$$

Direct observation count, mean and maximum fraud probability remain diagnostics and training evidence only. They are not inserted directly into the final formula.

In [3]:
knn_columns = [
    'merchant_abn', 'merchant_name', 'knn_score',
    'knn_mean_neighbor_distance', 'knn_confidence_score',
    'knn_out_of_distribution',
    'knn_merchant_risk_percentile', 'fraud_risk_index',
    'risk_safety_score'
]
knn_profile = profile.loc[profile.has_knn_merchant_risk_information, knn_columns]
display(knn_profile.sort_values('knn_merchant_risk_percentile', ascending=False).head(10))

,merchant_abn,merchant_name,knn_score,knn_mean_neighbor_distance,knn_confidence_score,knn_out_of_distribution,knn_merchant_risk_percentile,fraud_risk_index,risk_safety_score
3828,87725508705,NaN,0.706360,3.539514,0.032787,True,1.000000,0.945162,5.483798
3333,77126808895,NaN,0.704811,3.552671,0.032787,True,0.999774,0.873329,12.667099
1774,45636462354,Amet Consectetuer Limited,0.704548,3.484953,0.032787,True,0.999548,0.662815,33.718518
1922,48597174605,NaN,0.704244,3.486947,0.032787,True,0.999321,0.893044,10.695610
4100,93238141799,Integer Sem Corp.,0.700395,3.687119,0.032787,True,0.999095,0.700545,29.945533
291,15971734188,Magna Nec Quam LLP,0.697369,1.595681,0.377049,False,0.998869,0.555632,44.436788
3553,81906511933,Sodales Purus In Ltd,0.697083,3.622596,0.032787,True,0.998643,0.661116,33.888388
3751,86145109204,Non Quam Incorporated,0.696757,1.718026,0.327869,False,0.998417,0.898257,10.174341
599,22168725907,Sed Neque Limited,0.696419,4.437837,0.032787,True,0.998190,0.979721,2.027935
1141,32598536052,Feugiat Lorem Ipsum Industries,0.695354,4.235223,0.032787,True,0.997964,0.952868,4.713162


## 3. Combined fraud risk

The baseline gives equal weight to the consumer-exposure percentile and KNN merchant-risk percentile:

$$FraudRisk_m = \begin{cases}0.5C_m^* + 0.5K_m, & C,K\text{ both available}\\C_m^*, & C\text{ only}\\K_m, & K\text{ only}\end{cases}$$

KNN covers all eligible merchants; eight merchants without consumer exposure therefore use the KNN percentile alone. No merchant ranking is calculated here.

In [4]:
knn_scores = pd.read_csv(OUTPUT_DIR / 'knn_fraud_risk_scores.csv')
display(knn_scores.describe().T, knn_scores.head())

,count,mean,std,min,25%,50%,75%,max
merchant_abn,4422.0,5.467285e+10,2.594943e+10,1.002328e+10,3.182786e+10,5.477366e+10,7.671662e+10,9.999054e+10
consumer_exposure_percentile,4414.0,5.000000e-01,2.887733e-01,0.000000e+00,2.500000e-01,5.000000e-01,7.500000e-01,1.000000e+00
knn_score,4422.0,5.475279e-01,1.397450e-01,1.861083e-01,4.439436e-01,6.330051e-01,6.586069e-01,7.063598e-01
knn_merchant_risk_percentile,4422.0,5.000000e-01,2.887731e-01,0.000000e+00,2.500000e-01,5.000000e-01,7.500000e-01,1.000000e+00
knn_mean_neighbor_distance,4422.0,2.917926e+00,1.229611e+00,3.297420e-01,1.854854e+00,2.960798e+00,3.942012e+00,7.666675e+00
knn_max_neighbor_distance,4422.0,3.715074e+00,1.480242e+00,5.894317e-01,2.523629e+00,3.763858e+00,4.762435e+00,1.003973e+01
knn_distance_percentile,4422.0,8.082353e-01,1.965056e-01,1.639344e-02,6.885246e-01,9.016393e-01,9.672131e-01,1.000000e+00
knn_confidence_score,4422.0,1.917647e-01,1.965056e-01,0.000000e+00,3.278689e-02,9.836066e-02,3.114754e-01,9.836066e-01
fraud_risk_index,4422.0,4.996878e-01,2.108741e-01,2.037994e-03,3.380964e-01,5.043459e-01,6.513246e-01,9.861793e-01
risk_safety_score,4422.0,5.003122e+01,2.108741e+01,1.382075e+00,3.486754e+01,4.956541e+01,6.619036e+01,9.979620e+01


,merchant_abn,merchant_name,consumer_exposure_percentile,knn_score,knn_merchant_risk_percentile,knn_mean_neighbor_distance,knn_max_neighbor_distance,knn_distance_percentile,knn_confidence_score,knn_out_of_distribution,knn_score_source,fraud_risk_index,risk_safety_score
0,10023283211,Felis Limited,0.579878,0.509525,0.334314,4.486615,5.220752,0.967213,0.032787,True,full_model_for_unlabelled_merchant,0.457096,54.290443
1,10142254217,Arcu Ac Orci Corporation,0.447315,0.657310,0.692830,3.447449,4.239421,0.967213,0.032787,True,full_model_for_unlabelled_merchant,0.570072,42.992779
2,10165489824,Nunc Sed Company,0.000000,0.313441,0.111061,3.130178,4.717325,0.934426,0.065574,False,full_model_for_unlabelled_merchant,0.055530,94.446958
3,10187291046,Ultricies Dignissim Lacus Foundation,0.114888,0.658932,0.765890,1.129005,1.470959,0.426230,0.573770,False,full_model_for_unlabelled_merchant,0.440389,55.961105
4,10192359162,Enim Condimentum PC,0.902334,0.652009,0.566614,1.489107,2.008204,0.622951,0.377049,False,full_model_for_unlabelled_merchant,0.734474,26.552605


## 4. Interpretation and limitations

- `fraud_risk_index` is a relative risk index, **not** a fraud probability.
- Only 61 merchants have direct merchant fraud observations.
- The KNN merchant score is learned from only 61 directly observed merchants, so uncertainty remains high.
- Consumer exposure uses model-estimated consumer risk and is not proof of merchant wrongdoing.
- KNN model-score coverage and raw consumer-label coverage are reported separately; the former can be complete even when direct label evidence is sparse.
- Coverage, direct observation count, and maximum observed risk must accompany the index as reliability diagnostics.
- Eight merchants have KNN-only risk because consumer exposure is unavailable; no merchant is assigned zero merely because evidence is missing.